In [1]:
import torch
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score
from collections import defaultdict 
from poincare import PoincareManifold          # tes fichiers locaux
from model import Distance_PE
from data import G_hpo

In [26]:
# Modifier à chaque fois :
checkpoint = torch.load('logs/2026_5_11/6/model_final.pt', map_location='cpu', weights_only=False)

objects = checkpoint['objects']
node2id = checkpoint['node2id']
losses = checkpoint['losses']
norm_history = checkpoint['norm_history']
edges_closed = checkpoint['edges']
data = checkpoint['data']
hp = checkpoint['hyperparams']

edges = np.array([(node2id[u], node2id[v]) for u, v in G_hpo.edges()],dtype=np.int64)


In [27]:
manifold = PoincareManifold()
model = Distance_PE(n=len(objects), dim=hp['dim'],
                       manifold=manifold, sparse=False, learn_curvature=False, init_curvature=1., weight_decay=0)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

Distance_PE(
  (embeddings): Embedding(19389, 15)
)

In [28]:
W = model.weight.detach().cpu().numpy()   # (N, dim)
norms = np.linalg.norm(W, axis=1)

print(f"Modèle chargé — {len(objects)} nœuds | dim={hp['dim']} | "
      f"{len(losses)} epochs")
print(f"Norme moy={norms.mean():.4f} | max={norms.max():.4f}")

i_min = np.argmin(norms).item()
print(f"Index de la plus petite norme : {i_min}")
print(f"Position du point : {W[i_min]}")


Modèle chargé — 19389 nœuds | dim=15 | 519 epochs
Norme moy=0.9853 | max=0.9900
Index de la plus petite norme : 101
Position du point : [ 0.08208764  0.07856129 -0.02699255  0.00412983  0.00606092 -0.00449228
  0.03793498 -0.01691381 -0.03460361 -0.02982806  0.01132905 -0.0086228
  0.00574168  0.02016685  0.01095275]


In [29]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])
norms = model.weight.detach().norm(dim=-1).numpy()

# Corrélation degré/norme attendue : négative
rho, pval = spearmanr(degrees, norms)
print(f"Corrélation Spearman degré/norme : {rho:.3f} (p={pval:.2e})")

Corrélation Spearman degré/norme : 0.160 (p=6.37e-111)


In [30]:
pos_neighbors = defaultdict(set)
pos_parents = defaultdict(set)

for u, v in edges:
    pos_neighbors[int(u)].add(int(v))
    pos_parents[int(v)].add(int(u))
    
len(pos_parents[i_min])
len(pos_parents[0])

7

In [31]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])

# Top 10 plus proches du centre
center_ids = np.argsort(norms)[:10]
print("=== 10 nœuds les plus proches du CENTRE ===")
for i in center_ids:
    print(f"  {objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

# Top 10 plus proches du bord
border_ids = np.argsort(norms)[-10:]
print("\n=== 10 nœuds les plus proches du BORD ===")
for i in border_ids:
    print(f"{objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

=== 10 nœuds les plus proches du CENTRE ===
  HP:0000118                     norme=0.1352  degré=1
  HP:0033127                     norme=0.3060  degré=2
  HP:0001939                     norme=0.3075  degré=2
  HP:0000001                     norme=0.3709  degré=0
  HP:0010766                     norme=0.4476  degré=3
  HP:0000707                     norme=0.4745  degré=2
  HP:0001574                     norme=0.4877  degré=2
  HP:0000924                     norme=0.5002  degré=3
  HP:0011805                     norme=0.5182  degré=3
  HP:0012649                     norme=0.5319  degré=3

=== 10 nœuds les plus proches du BORD ===
HP:0001436                     norme=0.9900  degré=5
HP:0033993                     norme=0.9900  degré=7
HP:0009681                     norme=0.9900  degré=15
HP:6000792                     norme=0.9900  degré=3
HP:0009391                     norme=0.9900  degré=14
HP:0006045                     norme=0.9900  degré=3
HP:0100273                     norme=0.9900

In [8]:
@torch.no_grad()
def evaluate(model, objects, edges, node2id):
    model.eval()
    W = model.weight.to(device)
    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))

    ranksum, ap_scores = 0, 0
    nranks = 0
    iters = 0
    labels = np.empty(model.embeddings.weight.size(0))
 
    for u in tqdm(objects):
        labels.fill(0)
        u = int(node2id[u])
        neighbors = pos_neighbors.get(u, set())
        if not neighbors :
            continue
        u_exp = W[u].unsqueeze(0).expand(W.shape[0], -1)  # Coordonnées de u dans la boule de Poincaré
        dists = manifold.distance(u_exp, W, 1).numpy()  # Distance de Poincaré de u aux autres noeuds
        dists[u] = 1e12
        #order = np.argsort(dists)  # Tri par distance décroissante p/r à u
        sorted_ind = np.argsort(dists)

        #ranks = int(np.where(order == v)[0][0]) + 1  # Rang du noeud v p/r à u dans l'embedding
        #ranks.append(rank)
        ranks, = np.where(np.isin(sorted_ind, list(neighbors)))
        ranks += 1
        N = ranks.shape[0]

        ranksum += ranks.sum() - (N * (N - 1) / 2)
        nranks += ranks.shape[0]
        labels[list(neighbors)] = 1
        ap_scores += average_precision_score(labels, -dists)
        iters += 1

        #pos  = pos_neighbors[u]  # Voisins de u dans la représentation initiale
        #hits, psum = 0, 0.0 
        #for k, idx in enumerate(order[1:], 1):  # On parcourt les noeuds du plus proche au plus éloigné
            #if idx in pos:
                #hits  += 1
                #psum  += hits / k
        #aps.append(psum / max(len(pos), 1))
 
    return float(ranksum), nranks, ap_scores, iters

In [32]:
objects_hpo = list(G_hpo.nodes())
node2id_hpo = {n: i for i, n in enumerate(objects_hpo)}
edges_hpo = np.array([(node2id_hpo[v], node2id_hpo[u]) for u, v in G_hpo.edges()],dtype=np.int64)
new_edges = [(v, u) for u, v in edges]
results = evaluate(model.weight.detach().cpu(), objects, new_edges, node2id)

print("Erreur moyenne sur le rang : ", float(results[0]) / results[1])
print("Mean Average Precision :", float(results[2]) / results[3])

AttributeError: 'Tensor' object has no attribute 'eval'

In [24]:
@torch.no_grad()
def evaluate2(model, objects, edges, node2id, device):
    """
    Retourne MAP et mean rank.
    Corrige : distance avec model.c, calcul GPU, pas de fuite u dans le ranking.
    """
    model.eval()
    W = model.weight.to(device)  # (N, dim)

    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))

    ap_scores = []
    ranks_all = []
    N = W.shape[0]
    labels = np.zeros(N)

    for obj in tqdm(objects):
        u = int(node2id[obj])
        neighbors = pos_neighbors.get(u, set())
        if not neighbors:
            continue

        # Distances GPU avec la bonne courbure
        u_emb = W[u].unsqueeze(0).expand(N, -1)   # (N, dim)
        dists = model.manifold.distance(u_emb, W, model.c)  # (N,)
        dists[u] = float('inf')                    # exclure u lui-même
        dists_np = dists.cpu().numpy()

        max_finite = dists_np[np.isfinite(dists_np)].max()
        dists_np[~np.isfinite(dists_np)] = max_finite + 1.0

        # Rang des voisins
        sorted_ind = np.argsort(dists_np)
        ranks = np.where(np.isin(sorted_ind, list(neighbors)))[0] + 1
        # Correction : soustraire les rangs des autres voisins placés avant
        n_neighbors = len(neighbors)
        corrected_ranks = ranks - np.arange(n_neighbors)
        ranks_all.extend(corrected_ranks.tolist())

        # AP
        labels.fill(0)
        labels[list(neighbors)] = 1
        ap_scores.append(average_precision_score(labels, -dists_np))

    map_score  = float(np.mean(ap_scores))
    mean_rank  = float(np.mean(ranks_all))

    model.train()
    return map_score, mean_rank

In [33]:
results = evaluate2(model, objects, new_edges, node2id, device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))

results

  0%|          | 0/19389 [00:00<?, ?it/s]

100%|██████████| 19389/19389 [00:31<00:00, 611.01it/s] 


(0.6554251339418409, 176.15187734932636)

In [19]:
print("min", norms.min(), "moy", norms.mean(), "max", norms.max())

min 0.05153147 moy 0.35499746 max 0.5878738
